In [21]:
import sys
import requests
import json
import os
import traceback
from bs4 import BeautifulSoup
import re
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, FloatType, DoubleType
from pyspark.sql.functions import col
from datetime import datetime

# Get the current local date and time
current_datetime = datetime.now()
current_datetime_str = current_datetime.strftime("%Y-%m-%d %H:%M:%S")
print(current_datetime_str)


spark = SparkSession.builder.appName("StockDataFrame").getOrCreate()

# sector = "Consumer Discretionary"
# url = "https://www.tradingview.com/symbols/SP-S5CONS/components/"
# file_path = "C:/Projects/testprofile.csv"

try:
    # sector = sys.argv[1]
    # url = sys.argv[2]
    # file_path = sys.argv[3]

    sector = "Consumer Discretionary"
    url = "https://www.tradingview.com/symbols/SP-S5CONS/components/"
    # file_path = "C:/Projects/testprofile.csv"

    timeout_seconds = 5
    req = requests.get(url, timeout=timeout_seconds)
    req_content = req.text

    html = req.text
    soup = BeautifulSoup(html, 'html.parser')

    field_list=[]
    for tag_td in soup.select("th[class*='cell-']"):
        field = tag_td.text.replace(".", "").replace(" ", "_") \
            .replace("_", "").replace("%", "Pct").replace("/", "").replace("(", "") \
            .replace(")", "").replace("*", "").replace('\xa0', ' ')        
        field_list.append(field)
        
    # Add the field ImportDatetime
    field_list.append("ImportDatetime")

    schema = StructType([
        StructField(field, eval('StringType')(), True)
        for field in field_list
    ])
    # print(schema)

    # Each record is a tuple, record_tuple_list is a collection of records
    record_tuple_list=[]
    for tag_tr in soup.select("tr[class*=' listRow']"):
        tag_a = tag_tr.find('a', attrs={'class': re.compile('^tickerName.*')})
        ticker = tag_a.text
        record = ""
        record_list=[ticker]
        for tag_td in tag_tr.select("td[class*='cell-']"):
            if tag_td.text.find(ticker) < 0:
                record_list.append(tag_td.text.replace("\u202f", ""))
        record_list.append(current_datetime_str)
        # Tuple is immutable in Python, so we cannot append an element to a tuple. 
        # This is why we need to build up the list first then convert it to a tuple.
        record_tuple= tuple(record_list)
        record_tuple_list.append(record_tuple)

    # print(record_tuple_list)    
    df = spark.createDataFrame(record_tuple_list, schema)
    df.select("symbol", "Sector","ImportDatetime").show()   

except:
    exc_type, exc_value, exc_traceback = sys.exc_info()
    exceptMessage = repr(traceback.format_exception(exc_type, exc_value, exc_traceback))
    message = "Error(-1): The data cannot be downloaded. <Except Message: " + exceptMessage + "> <Quote URL: "
    print(message)


2025-08-12 18:27:58
StructType([StructField('Symbol', StringType(), True), StructField('Marketcap', StringType(), True), StructField('Price', StringType(), True), StructField('Change Pct', StringType(), True), StructField('Volume', StringType(), True), StructField('RelVolume', StringType(), True), StructField('PE', StringType(), True), StructField('EPSdilTTM', StringType(), True), StructField('EPSdilgrowthTTMYoY', StringType(), True), StructField('Divyield PctTTM', StringType(), True), StructField('Sector', StringType(), True), StructField('AnalystRating', StringType(), True), StructField('ImportDatetime', StringType(), True)])
+------+--------------------+-------------------+
|symbol|              Sector|     ImportDatetime|
+------+--------------------+-------------------+
|   WMT|        Retail trade|2025-08-12 18:27:58|
|  COST|        Retail trade|2025-08-12 18:27:58|
|    PG|Consumer non-dura...|2025-08-12 18:27:58|
|    KO|Consumer non-dura...|2025-08-12 18:27:58|
|    PM|Consum

In [5]:
import yaml

# Define the key hierarchy
key_hierarchy = ["server", "port"]

# Read the YAML file
try:
    with open('test.yaml', 'r') as file:
        data = yaml.safe_load(file)
    
    # Access the port number using the key hierarchy
    value = data
    for key in key_hierarchy:
        value = value.get(key)
        if value is None:
            raise KeyError(f"Key '{key}' not found or value is None")
    
    # Print the port number
    print(f"Server port: {value}")

except FileNotFoundError:
    print("Error: config.yaml not found")
except yaml.YAMLError as e:
    print(f"Error parsing YAML: {e}")
except KeyError as e:
    print(f"Error: {e}")

Server port: 8080
